In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

# Configuration
BRONZE_TABLE = "workspace.bronze.bronze_sales_transactions"
SILVER_SCHEMA = "workspace.silver"
CURRENT_TIMESTAMP = F.current_timestamp()

In [0]:
# Read bronze data
bronze_df = spark.table(BRONZE_TABLE)


print(f"Total records in bronze: {bronze_df.count():,}")
display(bronze_df.limit(5))

In [0]:
# Apply data quality transformations
silver_cleaned_df = (
    bronze_df
    # Remove duplicates based on transaction_id
    .dropDuplicates(["transaction_id"])
    
    # Filter out invalid records
    .filter(
        (F.col("transaction_id").isNotNull()) &
        (F.col("customer_id").isNotNull()) &
        (F.col("product_id").isNotNull()) &
        (F.col("total_amount") >= 0) &
        (F.col("quantity") > 0)
    )
    
    # Standardize data types and formats
    .withColumn("transaction_date", F.to_date(F.col("transaction_date")))
    .withColumn("customer_signup_date", F.to_date(F.col("customer_signup_date")))
    .withColumn("customer_email", F.lower(F.trim(F.col("customer_email"))))
    .withColumn("customer_first_name", F.initcap(F.trim(F.col("customer_first_name"))))
    .withColumn("customer_last_name", F.initcap(F.trim(F.col("customer_last_name"))))
    
    # Calculate derived fields
    .withColumn("discount_amount", F.col("unit_price") * F.col("quantity") * F.col("discount_pct") / 100)
    .withColumn("subtotal", F.col("unit_price") * F.col("quantity"))
    
    # Add processing metadata
    .withColumn("silver_processing_timestamp", CURRENT_TIMESTAMP)
)

print(f"Records after data quality: {silver_cleaned_df.count():,}")
display(silver_cleaned_df.limit(5))

In [0]:
# Create customer dimension table
silver_customers = (
    silver_cleaned_df
    .select(
        "customer_id",
        "customer_first_name",
        "customer_last_name",
        "customer_email",
        "customer_city",
        "customer_province",
        "customer_segment",
        "customer_signup_date"
    )
    .dropDuplicates(["customer_id"])
    .withColumn("customer_full_name", 
                F.concat_ws(" ", F.col("customer_first_name"), F.col("customer_last_name")))
    .withColumn("updated_at", CURRENT_TIMESTAMP)
)

print(f"Unique customers: {silver_customers.count():,}")
display(silver_customers.limit(5))

In [0]:
# Create product dimension table
silver_products = (
    silver_cleaned_df
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "unit_cost",
        "unit_price"
    )
    .dropDuplicates(["product_id"])
    .withColumn("profit_margin", 
                F.round((F.col("unit_price") - F.col("unit_cost")) / F.col("unit_price") * 100, 2))
    .withColumn("updated_at", CURRENT_TIMESTAMP)
)

print(f"Unique products: {silver_products.count():,}")
display(silver_products.limit(5))

In [0]:
# Create store dimension table
silver_stores = (
    silver_cleaned_df
    .select(
        "store_id",
        "store_name",
        "store_city",
        "store_province",
        "store_type"
    )
    .dropDuplicates(["store_id"])
    .withColumn("updated_at", CURRENT_TIMESTAMP)
)

print(f"Unique stores: {silver_stores.count():,}")
display(silver_stores.limit(5))

In [0]:
# Create transactions fact table
silver_transactions = (
    silver_cleaned_df
    .select(
        "transaction_id",
        "transaction_date",
        "transaction_time",
        "customer_id",
        "product_id",
        "store_id",
        "quantity",
        "unit_price",
        "discount_pct",
        "discount_amount",
        "subtotal",
        "total_amount",
        "payment_method",
        "ingestion_timestamp",
        "silver_processing_timestamp"
    )
)

print(f"Total transactions: {silver_transactions.count():,}")
display(silver_transactions.limit(5))

In [0]:
# Write dimension tables to silver schema
silver_customers.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.dim_customers")
print("✓ Wrote dim_customers")

silver_products.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.dim_products")
print("✓ Wrote dim_products")

silver_stores.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.dim_stores")
print("✓ Wrote dim_stores")

# Write fact table to silver schema
silver_transactions.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA}.fact_transactions")
print("✓ Wrote fact_transactions")

print("\n" + "="*50)
print("Silver layer ingestion completed successfully!")
print("="*50)

In [0]:
# Generate data quality summary
print("SILVER LAYER SUMMARY")
print("="*50)
print(f"Customers: {silver_customers.count():,}")
print(f"Products: {silver_products.count():,}")
print(f"Stores: {silver_stores.count():,}")
print(f"Transactions: {silver_transactions.count():,}")
print("="*50)

# Quick validation queries
print("\nSample joined data:")
validation_df = (
    spark.table(f"{SILVER_SCHEMA}.fact_transactions")
    .join(spark.table(f"{SILVER_SCHEMA}.dim_customers"), "customer_id")
    .join(spark.table(f"{SILVER_SCHEMA}.dim_products"), "product_id")
    .join(spark.table(f"{SILVER_SCHEMA}.dim_stores"), "store_id")
    .select(
        "transaction_id",
        "transaction_date",
        "customer_full_name",
        "product_name",
        "store_name",
        "total_amount"
    )
)

display(validation_df.limit(10))